# 08_pipeline.ipynb

## Sprint 2 - PIPE: Integración del pipeline reproducible

En este notebook se integran las transformaciones desarrolladas en el Sprint 2 en un flujo reproducible usando `ColumnTransformer` y `Pipeline`.

### Objetivo
Construir y validar un pipeline reutilizable para transformar los datos de forma consistente en entrenamiento, prueba y datos nuevos.

### Rol en el Sprint 2
Como Pipeline Builder, este notebook:
- integra las transformaciones previas,
- valida el flujo con `fit` y `transform`,
- persiste el pipeline con `joblib`,
- genera un dataset final procesado para las siguientes etapas.

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
import pandas as pd

from src.preprocessing import (
    split_data,
    get_column_groups,
    build_preprocessor,
    save_preprocessor,
)

## Carga de datos

Se carga el dataset base desde `data/raw/`. Este dataset será transformado dentro del pipeline sin modificar el archivo original.

In [3]:
df = pd.read_csv("../data/raw/10-employee_performance.csv")
df.head()

,employee_id,age,gender,education,department,job_level,years_at_company,years_in_current_role,monthly_salary,overtime_hours_monthly,num_projects_completed,performance_rating,training_hours_yearly,work_life_balance_score,distance_from_home_km,num_companies_worked,job_satisfaction,relationship_with_manager,stock_option_level,attrition
0,1,38,Female,Master,IT,3,6.3,0.2,6072.54,11,7,2,39.0,5,4.4,3,3,5,3,0
1,2,32,Male,Bachelor,Finance,1,3.4,3.8,3058.51,15,6,3,57.1,4,4.5,2,4,4,2,0
2,3,24,Female,Bachelor,Engineering,1,0.8,2.9,9270.48,5,6,5,41.1,1,49.6,2,3,1,0,1
3,4,33,Male,Bachelor,Sales,5,0.2,1.6,7567.36,3,8,3,21.2,2,0.7,2,5,4,1,0
4,5,51,Male,Bachelor,Finance,3,3.8,7.1,5621.87,5,8,4,42.6,4,6.6,3,3,4,2,1


## Ajustes iniciales

Se elimina `employee_id` porque es un identificador y no aporta valor predictivo al pipeline.
Además, se define la variable objetivo (`target`) usando el nombre exacto presente en el dataset.

In [4]:
df = df.drop(columns=["employee_id"])
target = "performance_rating"
df.columns

Index(['age', 'gender', 'education', 'department', 'job_level',
       'years_at_company', 'years_in_current_role', 'monthly_salary',
       'overtime_hours_monthly', 'num_projects_completed',
       'performance_rating', 'training_hours_yearly',
       'work_life_balance_score', 'distance_from_home_km',
       'num_companies_worked', 'job_satisfaction', 'relationship_with_manager',
       'stock_option_level', 'attrition'],
      dtype='str')

## División Train/Test

El train/test split se realiza antes de cualquier transformación para evitar data leakage.
Toda transformación que aprende parámetros del dato debe ajustarse únicamente con el conjunto de entrenamiento. :contentReference[oaicite:2]{index=2}

In [5]:
X_train, X_test, y_train, y_test = split_data(df, target)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (11200, 18)
X_test: (2800, 18)
y_train: (11200,)
y_test: (2800,)


In [6]:
y_train.value_counts()

performance_rating
3    4574
4    3353
2    1669
5    1035
1     569
Name: count, dtype: int64

## Identificación de columnas

Se separan automáticamente las variables numéricas y categóricas para aplicar transformaciones distintas mediante `ColumnTransformer`.

In [7]:
num_cols, cat_cols = get_column_groups(X_train)

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

Numéricas: ['age', 'job_level', 'years_at_company', 'years_in_current_role', 'monthly_salary', 'overtime_hours_monthly', 'num_projects_completed', 'training_hours_yearly', 'work_life_balance_score', 'distance_from_home_km', 'num_companies_worked', 'job_satisfaction', 'relationship_with_manager', 'stock_option_level', 'attrition']
Categóricas: ['gender', 'education', 'department']


## Construcción del pipeline

Se construye un preprocesador con:
- imputación y escalado para variables numéricas,
- imputación y One-Hot Encoding para variables categóricas.

Esto permite encapsular las transformaciones en un objeto reproducible y reutilizable. :contentReference[oaicite:3]{index=3}

In [8]:
preprocessor = build_preprocessor(num_cols, cat_cols)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed:", X_train_processed.shape)
print("X_test_processed:", X_test_processed.shape)

X_train_processed: (11200, 30)
X_test_processed: (2800, 30)


## Validación del preprocesamiento

Se verifica que el pipeline puede ajustarse con `fit_transform` sobre train y aplicarse con `transform` sobre test sin errores, cumpliendo el criterio esperado para el Sprint 2. :contentReference[oaicite:4]{index=4}

In [10]:
X_train_df = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed
)

X_train_df.head()

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.393240,0.490751,-0.652176,0.522389,0.405670,0.357835,-0.802618,-0.534814,0.715336,0.916668,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.215748,1.290579,-0.944830,-0.078210,-0.998118,1.066732,-2.431562,-1.452277,1.630130,1.846434,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,1.437220,-0.309076,-0.652176,1.363227,-1.050657,-0.351062,-0.802618,-0.990197,0.715336,0.013278,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.350222,2.090407,-0.854782,0.001870,0.029454,0.003386,2.455269,-0.581692,-0.199458,0.553993,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-1.520723,-1.108904,-0.337010,-0.638769,-1.145357,0.357835,-0.802618,0.329074,-0.199458,-0.599972,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## Generación del dataset final

Se genera un dataset procesado final a partir de la transformación del conjunto completo de variables explicativas usando el preprocesador ya ajustado sobre train.

Este archivo se guarda en `data/processed/` para su versionado con DVC, tal como exige el Sprint 2. :contentReference[oaicite:5]{index=5}

In [11]:
os.makedirs("../data/processed", exist_ok=True)

feature_names = preprocessor.get_feature_names_out()

X_all = df.drop(columns=[target])
X_all_processed = preprocessor.transform(X_all)

if hasattr(X_all_processed, "toarray"):
    X_all_processed = X_all_processed.toarray()

final_df = pd.DataFrame(X_all_processed, columns=feature_names)
final_df[target] = df[target].values

final_df.to_csv("../data/processed/final_dataset.csv", index=False)

print("final_dataset.csv guardado")
final_df.head()

final_dataset.csv guardado


,num__age,num__job_level,num__years_at_company,num__years_in_current_role,num__monthly_salary,num__overtime_hours_monthly,num__num_projects_completed,num__training_hours_yearly,num__work_life_balance_score,num__distance_from_home_km,...,cat__education_PhD,cat__department_Engineering,cat__department_Finance,cat__department_HR,cat__department_IT,cat__department_Legal,cat__department_Marketing,cat__department_Operations,cat__department_Sales,performance_rating
0,-0.302746,0.490751,0.405880,-0.919049,0.350776,1.066732,0.419089,-0.052644,1.630130,-0.698883,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2
1,-0.824736,-1.108904,-0.246963,0.522389,-1.019776,2.484526,0.011853,1.159478,0.715336,-0.692289,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3
2,-1.520723,-1.108904,-0.832270,0.162029,1.804956,-1.059960,0.011853,0.087989,-2.029045,2.281644,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
3,-0.737738,2.090407,-0.967341,-0.358490,1.030506,-1.768857,0.826325,-1.244676,-1.114252,-0.942864,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3
4,0.828232,0.490751,-0.156916,1.843706,0.145845,-1.059960,0.826325,0.188441,0.715336,-0.553813,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4


## Persistencia del pipeline

El pipeline se serializa con `joblib` para reutilizar exactamente las mismas transformaciones en los siguientes sprints. El artefacto se guarda en `models/`. :contentReference[oaicite:6]{index=6}

In [12]:
os.makedirs("../models", exist_ok=True)

save_preprocessor(preprocessor, "../models/preprocessing_pipeline.pkl")
print("preprocessing_pipeline.pkl guardado")

preprocessing_pipeline.pkl guardado


## Conclusión

Se construyó un pipeline de preprocesamiento reproducible utilizando `ColumnTransformer`, aplicando transformaciones diferenciadas para variables numéricas y categóricas. El flujo respeta el split previo al fit, genera un dataset final procesado y persiste el pipeline para reutilización en etapas posteriores del proyecto.